In [0]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

constructors_schema = StructType(fields=
                             [
                             StructField("constructorId", StringType(), False),
                             StructField("constructorRef", StringType(), True),
                             StructField("name", StringType(), True),
                             StructField("nationality", StringType(), True),
                             StructField("url", StringType(), True)
                             ])


In [0]:
constructors_df=spark.read\
    .schema(constructors_schema)\
        .json("abfss://demofiles@formula1adls.dfs.core.windows.net/source_files/constructors.json")


In [0]:
constructors_df.display()

In [0]:
constructors_df.printSchema()

In [0]:
from pyspark.sql.functions import col
df=constructors_df.withColumn("constructorId", col("constructorId").cast(IntegerType()))

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp,current_date
v= constructors_df.withColumn("ingestion_timestamp", current_timestamp())\
    .withColumn("ingestion_date", current_date())\
        .withColumnRenamed("constructorId", "constructor_id")\
            .withColumnRenamed("constructorRef", "constructor_ref")\
                .drop("url")


In [0]:
v.display()

In [0]:
v.write.mode("overwrite").format("delta").option("path", "abfss://raw@formula1adls.dfs.core.windows.net/constructors").saveAsTable(f"formula1_{env}.bronze.constructors")

In [0]:
df=spark.table(f"formula1_{env}.bronze.constructors")
df.display()